# Exploration des données — M@ieutic

Corpus : thèses françaises soutenues/en cours (2021–2026), sections CNU ciblées  
Source : API Thèses.fr

---

## 0. Imports

In [ ]:
import json
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option('display.max_colwidth', 80)
sns.set_theme(style='whitegrid', palette='muted')

DATA_PATH = '../data/data.json'

## 1. Chargement & aperçu général

In [ ]:
with open(DATA_PATH, encoding='utf-8') as f:
    raw = json.load(f)

df = pd.DataFrame(raw)

# Typage
df['annee'] = df['annee'].astype(int)
df['cnu'] = df['cnu'].astype(str).str.zfill(2)
df['nb_directeurs'] = df['directeurs'].apply(len)
df['co_encadrement'] = df['nb_directeurs'] > 1

print(f"Nombre de thèses : {len(df):,}")
print(f"Variables        : {list(df.columns)}")
df.head(3)

## 2. Données manquantes

In [ ]:
missing = {}
for col in df.columns:
    if df[col].dtype == object:
        n = df[col].apply(lambda x: x is None or x == [] or x == '').sum()
    else:
        n = df[col].isna().sum()
    missing[col] = n

miss_df = pd.DataFrame.from_dict(missing, orient='index', columns=['manquants'])
miss_df['%'] = (miss_df['manquants'] / len(df) * 100).round(1)
miss_df['présents'] = len(df) - miss_df['manquants']
display(miss_df.sort_values('manquants', ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
cols = miss_df.sort_values('%', ascending=True).index
ax.barh(cols, miss_df.loc[cols, '%'], color='steelblue')
ax.set_xlabel('% de valeurs manquantes')
ax.set_title('Complétude par variable')
for i, v in enumerate(miss_df.loc[cols, '%']):
    ax.text(v + 0.2, i, f'{v}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../visuals/completude_variables.png', dpi=150)
plt.show()

## 3. Distribution temporelle

In [ ]:
annee_counts = df.groupby('annee').size().reset_index(name='nb_theses')
display(annee_counts)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(annee_counts['annee'], annee_counts['nb_theses'], color='steelblue')
ax.set_xlabel('Année')
ax.set_ylabel('Nombre de thèses')
ax.set_title('Nombre de thèses par année (2021–2026)')
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
for x, y in zip(annee_counts['annee'], annee_counts['nb_theses']):
    ax.text(x, y + 5, str(y), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../visuals/theses_par_annee.png', dpi=150)
plt.show()

## 4. Répartition par section CNU

In [ ]:
CNU_LABELS = {
    '04': '04 – Science politique',
    '06': '06 – Sciences de gestion',
    '17': '17 – Philosophie',
    '18': '18 – Arts',
    '19': '19 – Sociologie',
    '20': '20 – Ethnologie',
    '70': "70 – Sciences de l'éducation",
    '71': '71 – Info-com',
    '72': '72 – Épistémologie',
}

cnu_counts = df['cnu'].value_counts().sort_index()
cnu_counts.index = cnu_counts.index.map(lambda x: CNU_LABELS.get(x, x))

fig, ax = plt.subplots(figsize=(10, 5))
cnu_counts.sort_values().plot.barh(ax=ax, color='teal')
ax.set_xlabel('Nombre de thèses')
ax.set_title('Thèses par section CNU')
for i, v in enumerate(cnu_counts.sort_values()):
    ax.text(v + 5, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../visuals/theses_par_cnu.png', dpi=150)
plt.show()

## 5. Normalisation des noms d'établissements

Certains établissements apparaissent sous plusieurs noms (renommages, encodages différents).  
On crée une colonne `etablissement_norm` pour regrouper les variantes.

In [ ]:
ETAB_NORM = {
    # --- Encodage différent ---
    'Université de Brest (2025-....)': 'Université de Brest',
    'Université de Brest (2025-….)':  'Université de Brest',
    'Brest':                          'Université de Brest',
    # --- Lille ---
    'Université de Lille (2018-2021)': 'Université de Lille',
    'Université de Lille (2022-....)': 'Université de Lille',
    # --- Rennes ---
    'Rennes 1':                         'Université de Rennes',
    'Université de Rennes (2023-....)': 'Université de Rennes',
    'Rennes 2':                         'Université Rennes 2',
    # --- Toulouse ---
    'Toulouse 1':                        'Université Toulouse Capitole',
    'Toulouse 2':                        'Université Toulouse - Jean Jaurès',
    'Toulouse 3':                        'Université Toulouse III - Paul Sabatier',
    'Université de Toulouse (2023-....)':'Université de Toulouse',
    # --- Montpellier ---
    'Montpellier':                             'Université de Montpellier',
    'Université de Montpellier (2022-....)':   'Université de Montpellier',
    'Montpellier 3':                           'Université Paul-Valéry Montpellier 3',
    'Université de Montpellier Paul-Valéry':   'Université Paul-Valéry Montpellier 3',
    # --- Saint-Étienne ---
    'Saint-Etienne':                                      'Université Jean Monnet Saint-Étienne',
    'Saint-Etienne, Université Jean Monnet (2025-....)':  'Université Jean Monnet Saint-Étienne',
    # --- Nîmes ---
    'Nîmes':           'Université de Nîmes',
    'Nîmes Université':'Université de Nîmes',
    # --- Casse ---
    'université Paris-Saclay': 'Université Paris-Saclay',
    # --- Dijon ---
    'Dijon, Université Bourgogne Europe': 'Université de Bourgogne',
}

df['etablissement_norm'] = df['etablissement'].map(ETAB_NORM).fillna(df['etablissement'])

avant = df['etablissement'].nunique()
apres = df['etablissement_norm'].nunique()
print(f'Avant normalisation : {avant} noms distincts')
print(f'Après normalisation : {apres} noms distincts')
print(f'Réduction           : -{avant - apres} variantes fusionnées')

In [ ]:
# Tableau des fusions effectuées
fusions = (
    df[df['etablissement'] != df['etablissement_norm']]
    .groupby(['etablissement', 'etablissement_norm'])
    .size()
    .reset_index(name='nb_theses')
    .sort_values('etablissement_norm')
)
display(fusions)

## 6. Établissements

In [ ]:
etab_counts = df['etablissement_norm'].value_counts()
print(f"Nombre d'établissements distincts (normalisés) : {etab_counts.nunique()}")
print()
print('Top 15 établissements :')
display(etab_counts.head(15).to_frame('nb_theses'))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
etab_counts.head(15).sort_values().plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('Nombre de thèses')
ax.set_title('Top 15 des établissements (noms normalisés)')
plt.tight_layout()
plt.savefig('../visuals/top15_etablissements.png', dpi=150)
plt.show()

top10_share = etab_counts.head(10).sum() / len(df) * 100
print(f'Les 10 premiers établissements représentent {top10_share:.1f}% des thèses')

## 7. Encadrement

In [ ]:
print('Distribution du nombre de directeurs par thèse :')
display(df['nb_directeurs'].value_counts().sort_index().to_frame('nb_theses'))

co_enc_pct = df['co_encadrement'].mean() * 100
print(f'\nTaux de co-encadrement : {co_enc_pct:.1f}%')

In [ ]:
dir_counter = Counter(d for row in df['directeurs'] for d in row)
top_dir = pd.Series(dir_counter).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
top_dir.sort_values().plot.barh(ax=ax, color='coral')
ax.set_xlabel('Nombre de thèses encadrées')
ax.set_title('Top 20 des directeurs de thèse')
plt.tight_layout()
plt.savefig('../visuals/top20_directeurs.png', dpi=150)
plt.show()

## 8. Accessibilité (Open Access)

In [ ]:
acc = df['accessible'].value_counts()
labels = ['Non accessible', 'Accessible (OA)']
print(acc)
print(f"\nTaux d'open access : {df['accessible'].mean()*100:.1f}%")

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(acc.values, labels=[labels[i] for i in acc.index.astype(int)],
       autopct='%1.1f%%', colors=['#d9534f', '#5cb85c'], startangle=90)
ax.set_title('Accessibilité des thèses')
plt.tight_layout()
plt.savefig('../visuals/accessibilite.png', dpi=150)
plt.show()

## 9. Géolocalisation

In [ ]:
geo_df = df.dropna(subset=['lat', 'lon'])
print(f"Thèses géolocalisées : {len(geo_df):,} / {len(df):,} ({len(geo_df)/len(df)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(geo_df['lon'], geo_df['lat'], alpha=0.3, s=10, color='steelblue')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Répartition géographique des établissements')
plt.tight_layout()
plt.savefig('../visuals/geo_scatter.png', dpi=150)
plt.show()

## 10. Synthèse

In [ ]:
print('=' * 52)
print('SYNTHÈSE DU CORPUS')
print('=' * 52)
print(f"Thèses totales              : {len(df):,}")
print(f"Période                     : {df['annee'].min()} – {df['annee'].max()}")
print(f"Sections CNU                : {df['cnu'].nunique()} sections")
print(f"Établissements (brut)       : {df['etablissement'].nunique()}")
print(f"Établissements (normalisés) : {df['etablissement_norm'].nunique()}")
print(f"Directeurs distincts        : {len(dir_counter):,}")
print(f"Taux co-encadrement         : {co_enc_pct:.1f}%")
print(f"Taux open access            : {df['accessible'].mean()*100:.1f}%")
print(f"Géolocalisation manquante   : {df['lat'].isna().sum():,} ({df['lat'].isna().mean()*100:.1f}%)")